# Check of the merged data (`CombinedData`)

Counts participants, trials and rows in every merged table, separately for each experiment.
The `.csv.gz` files are read with pandas, which reads **all** gzip members (files are written participant by participant);
gaze files are read in chunks so the check fits in memory.

Run this notebook from the `CombinedData` folder (or set `DATA` below).

In [ ]:
import gzip, os, time
import pandas as pd

DATA = r"C:\Users\barak\Documents\GitHub\Experiment_MW_Story\analysis\CombinedData"
os.chdir(DATA)
KEY = ['Experiment', 'Subject', 'TrialNum', 'Fragment']
print(sorted(f for f in os.listdir('.') if f.endswith(('.csv', '.csv.gz'))))

## 1. Behavioural table

In [ ]:
beh = pd.read_csv('behavioral.csv')
print(beh.shape)
summary = beh.groupby('Experiment').agg(participants=('Subject', 'nunique'), rows=('Subject', 'size'),
                                        with_thoughts=('has_thoughts', 'sum'))
summary['trials_per_participant'] = beh.groupby(['Experiment', 'Subject']).size().groupby('Experiment').agg(lambda s: sorted(s.unique()))
summary

## 2. Event tables (saccades, blinks, fixations)

In [ ]:
rows = []
for f in ['saccades', 'blinks', 'fixations']:
    ev = pd.read_csv(f'{f}.csv.gz', usecols=KEY)
    for exp, g in ev.groupby('Experiment'):
        rows.append(dict(file=f, Experiment=exp, participants=g.Subject.nunique(), rows=len(g),
                         trials_with_events=g.groupby('Subject').TrialNum.nunique().median()))
pd.DataFrame(rows)

## 3. Gaze tables (raw 500 Hz, preprocessed 100 Hz)

Read in chunks: per participant and trial we count samples. Takes about 1–2 min for `gaze_raw`.

In [ ]:
def gaze_summary(path, chunksize=5_000_000):
    t0, parts = time.time(), []
    for ch in pd.read_csv(path, usecols=['Experiment', 'Subject', 'TrialNum'],
                          dtype={'Experiment': 'category', 'Subject': 'category', 'TrialNum': 'int16'},
                          chunksize=chunksize):
        parts.append(ch.groupby(['Experiment', 'Subject', 'TrialNum'], observed=True).size())
    per_trial = pd.concat(parts).groupby(level=[0, 1, 2]).sum().rename('samples').reset_index()
    print(f'{path}: {per_trial.samples.sum():,} rows read in {time.time() - t0:.0f} s')
    return per_trial

def describe(per_trial):
    return per_trial.groupby('Experiment').agg(
        participants=('Subject', 'nunique'),
        rows=('samples', 'sum'),
        trials_per_participant=('TrialNum', lambda s: sorted(per_trial.loc[s.index].groupby('Subject').TrialNum.nunique().unique())))

raw  = gaze_summary('gaze_raw.csv.gz')
prep = gaze_summary('gaze_preproc.csv.gz')
display(describe(raw), describe(prep))

### Participants per experiment (gaze_raw)

In [ ]:
raw.groupby(['Experiment', 'Subject']).samples.sum().unstack(0)

### Raw vs preprocessed: same participants and trials, ~5x fewer samples

In [ ]:
m = raw.merge(prep, on=['Experiment', 'Subject', 'TrialNum'], how='outer', suffixes=('_raw', '_prep'), indicator=True)
print(m['_merge'].value_counts().to_dict())
print('raw/preproc sample ratio per trial: min %.3f, max %.3f' % ((m.samples_raw / m.samples_prep).min(), (m.samples_raw / m.samples_prep).max()))

## 4. Same participants in every table?

In [ ]:
sets = {'behavioral': beh[['Experiment', 'Subject']]}
for f in ['saccades', 'blinks', 'fixations']:
    sets[f] = pd.read_csv(f'{f}.csv.gz', usecols=['Experiment', 'Subject'])
sets['gaze_raw'], sets['gaze_preproc'] = raw, prep
ids = {k: set(map(tuple, v[['Experiment', 'Subject']].drop_duplicates().values)) for k, v in sets.items()}
ref = ids['behavioral']
for k, s in ids.items():
    print(f'{k:13s} participants: {len(s):3d} | missing vs behavioral: {sorted(ref - s)} | extra: {sorted(s - ref)}')

## 5. Why an archiver may show only 1–2 participants

Each file is a chain of gzip *members* (one per write). pandas/Python read all of them; some archivers (e.g. WinRAR)
unpack or report only one member, and Excel stops at 1,048,576 rows.

In [ ]:
def gzip_members(path):
    # count gzip member headers (0x1f 0x8b 0x08) by decompressing member by member
    import zlib
    n = 0
    with open(path, 'rb') as fh:
        data = fh.read(64 * 1024 * 1024)  # first 64 MB is enough to show the structure
    while data:
        d = zlib.decompressobj(16 + zlib.MAX_WBITS)
        d.decompress(data)
        n += 1
        data = d.unused_data
    return n

for f in ['saccades', 'blinks', 'fixations']:
    print(f, 'gzip members:', gzip_members(f'{f}.csv.gz'))